In [ ]:
from pathlib import Path
import shutil
import tarfile

from google.colab import drive

drive.mount("/content/drive")

bundle = Path("/content/digikala_inputs.tar.gz")
shutil.copyfile(
    "/content/drive/MyDrive/digikala-project-3/digikala_inputs.tar.gz",
    bundle,
)
with tarfile.open(bundle, "r:gz") as archive:
    archive.extractall("/content", filter="data")

print("Baseline inputs restored.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Baseline inputs restored.


In [ ]:
import csv
from pathlib import Path
from time import perf_counter

import joblib
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

from normalize_persian import normalize_text

train_path = Path("/content/digikala_splits/train_supervised.csv")
artifact_dir = Path("/content/digikala_artifacts")
artifact_dir.mkdir(exist_ok=True)

recommendation_labels = {
    "recommended": 0, "not_recommended": 1, "no_idea": 2, "": -1,
}
rating_labels = {"low": 0, "medium": 1, "high": 2, "": -1}
recommendation_targets = []
rating_targets = []

def training_texts():
    with train_path.open(encoding="utf-8", newline="") as file:
        for row in csv.DictReader(file):
            recommendation_targets.append(
                recommendation_labels[row["recommendation_status"]]
            )
            rating_targets.append(rating_labels[row["rating_band"]])
            yield normalize_text(row["body"])

tfidf = TfidfVectorizer(
    max_features=60_000,
    min_df=5,
    token_pattern=r"(?u)\b\w+(?:\u200c\w+)*\b",
    sublinear_tf=True,
    dtype=np.float32,
)

started = perf_counter()
X_train = tfidf.fit_transform(training_texts())
y_recommendation = np.asarray(recommendation_targets, dtype=np.int8)
y_rating_band = np.asarray(rating_targets, dtype=np.int8)
del recommendation_targets, rating_targets

assert X_train.shape[0] == len(y_recommendation) == len(y_rating_band)
assert np.all((y_recommendation >= 0) | (y_rating_band >= 0))

vectorizer_path = artifact_dir / "tfidf_vectorizer.joblib"
joblib.dump(tfidf, vectorizer_path)

matrix_mib = (
    X_train.data.nbytes + X_train.indices.nbytes + X_train.indptr.nbytes
) / 2**20
zero_rows = int(np.count_nonzero(np.diff(X_train.indptr) == 0))

print(f"Training rows: {X_train.shape[0]:,}")
print(f"TF-IDF vocabulary: {X_train.shape[1]:,}")
print(f"Sparse matrix size: {matrix_mib:,.1f} MiB")
print(f"Reviews without TF-IDF features: {zero_rows:,} ({zero_rows / X_train.shape[0]:.2%})")
print(f"Valid recommendation targets: {np.count_nonzero(y_recommendation >= 0):,}")
print(f"Valid rating-band targets: {np.count_nonzero(y_rating_band >= 0):,}")
print(f"Elapsed: {perf_counter() - started:,.1f} seconds")
print(f"Saved vectorizer: {vectorizer_path}")

Training rows: 3,570,765
TF-IDF vocabulary: 55,365
Sparse matrix size: 264.1 MiB
Reviews without TF-IDF features: 6,105 (0.17%)
Valid recommendation targets: 2,946,171
Valid rating-band targets: 3,570,759
Elapsed: 176.4 seconds
Saved vectorizer: /content/digikala_artifacts/tfidf_vectorizer.joblib


In [ ]:
import json
import warnings
from pathlib import Path
from time import perf_counter

import joblib
import numpy as np
import sklearn
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression

artifact_dir = Path("/content/digikala_artifacts")
reports_dir = Path("/content/digikala_reports")
reports_dir.mkdir(exist_ok=True)

settings = {
    "solver": "saga",
    "class_weight": "balanced",
    "C": 1.0,
    "tol": 1e-3,
    "max_iter": 60,
    "random_state": 2026,
}
training_report = {
    "sklearn_version": sklearn.__version__,
    "logistic_regression": settings,
    "heads": {},
}

for target, y, encoding in [
    ("recommendation_status", y_recommendation, recommendation_labels),
    ("rating_band", y_rating_band, rating_labels),
]:
    valid = y >= 0
    y_valid = y[valid]
    class_names = {index: label for label, index in encoding.items() if index >= 0}
    model = LogisticRegression(**settings)

    print(f"Training {target}: {len(y_valid):,} rows", flush=True)
    started = perf_counter()
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        model.fit(X_train[valid], y_valid)

    seconds = round(perf_counter() - started, 1)
    convergence_warnings = [
        str(item.message) for item in caught
        if issubclass(item.category, ConvergenceWarning)
    ]
    iterations = int(model.n_iter_.max())

    assert set(model.classes_) == set(class_names)
    assert np.allclose(model.predict_proba(X_train[:8]).sum(axis=1), 1)

    model_path = artifact_dir / f"tfidf_{target}.joblib"
    joblib.dump({"model": model, "class_names": class_names}, model_path)

    training_report["heads"][target] = {
        "training_rows": len(y_valid),
        "class_counts": {
            label: int(np.count_nonzero(y_valid == index))
            for index, label in class_names.items()
        },
        "iterations": iterations,
        "converged": not convergence_warnings,
        "convergence_warnings": convergence_warnings,
        "training_seconds": seconds,
        "artifact": model_path.name,
    }
    print(f"Iterations: {iterations} | Converged: {not convergence_warnings} | Time: {seconds}s")
    for message in convergence_warnings:
        print(message)
    print(f"Saved: {model_path}\n")

report_path = reports_dir / "baseline_training.json"
report_path.write_text(
    json.dumps(training_report, indent=2),
    encoding="utf-8",
)
print(f"Training report: {report_path}")

Training recommendation_status: 2,946,171 rows
Iterations: 60 | Converged: False | Time: 232.3s
The max_iter was reached which means the coef_ did not converge
Saved: /content/digikala_artifacts/tfidf_recommendation_status.joblib

Training rating_band: 3,570,759 rows
Iterations: 60 | Converged: False | Time: 258.1s
The max_iter was reached which means the coef_ did not converge
Saved: /content/digikala_artifacts/tfidf_rating_band.joblib

Training report: /content/digikala_reports/baseline_training.json


In [ ]:
import json
import shutil
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix

from normalize_persian import normalize_text

artifact_dir = Path("/content/digikala_artifacts")
reports_dir = Path("/content/digikala_reports")
validation_path = Path("/content/digikala_splits/validation.csv")

vectorizer = joblib.load(artifact_dir / "tfidf_vectorizer.joblib")
bundles = {
    target: joblib.load(artifact_dir / f"tfidf_{target}.joblib")
    for target in ["recommendation_status", "rating_band"]
}
encodings = {
    target: {**{name: code for code, name in bundle["class_names"].items()}, "": -1}
    for target, bundle in bundles.items()
}

actual = {target: [] for target in bundles}
predicted = {target: [] for target in bundles}
total_rows = 0

for chunk in pd.read_csv(
    validation_path, chunksize=50_000, dtype=str, keep_default_na=False
):
    total_rows += len(chunk)
    features = vectorizer.transform(normalize_text(text) for text in chunk["body"])

    for target, bundle in bundles.items():
        encoded = chunk[target].map(encodings[target])
        if encoded.isna().any():
            raise ValueError(f"Unexpected label in validation: {target}")
        y = encoded.to_numpy(dtype=np.int8)
        valid = y >= 0
        actual[target].append(y[valid])
        if valid.any():
            predicted[target].append(bundle["model"].predict(features[valid]))

results = {"split": "validation", "total_rows": total_rows, "heads": {}}

for target, bundle in bundles.items():
    y_true = np.concatenate(actual[target])
    y_pred = np.concatenate(predicted[target])
    codes = sorted(bundle["class_names"])
    names = [bundle["class_names"][code] for code in codes]

    metrics = classification_report(
        y_true, y_pred, labels=codes, target_names=names,
        output_dict=True, zero_division=0,
    )
    matrix = confusion_matrix(y_true, y_pred, labels=codes)
    balanced = float(balanced_accuracy_score(y_true, y_pred))
    record = {
        "evaluated_rows": int(len(y_true)),
        "missing_target_rows": int(total_rows - len(y_true)),
        "metrics": metrics,
        "balanced_accuracy": balanced,
        "confusion_matrix": {"class_order": names, "counts": matrix.tolist()},
    }

    print(f"\n{target} | evaluated rows: {len(y_true):,}")
    print(f"Accuracy: {metrics['accuracy']:.4f} | Balanced accuracy: {balanced:.4f}")
    print(pd.DataFrame(metrics).T.loc[names + ["macro avg"]].round(4).to_string())
    print("\nConfusion matrix (rows: actual, columns: predicted)")
    print(pd.DataFrame(matrix, index=names, columns=names).to_string())

    if target == "rating_band":
        distance = float(np.abs(y_true - y_pred).mean())
        record["mean_absolute_ordinal_distance"] = distance
        print(f"Mean absolute ordinal distance: {distance:.4f}")

    results["heads"][target] = record

report_path = reports_dir / "baseline_validation.json"
report_path.write_text(json.dumps(results, indent=2), encoding="utf-8")

project_drive = Path("/content/drive/MyDrive/digikala-project-3")
shutil.copytree(artifact_dir, project_drive / "artifacts", dirs_exist_ok=True)
shutil.copytree(reports_dir, project_drive / "reports", dirs_exist_ok=True)
print("\nBaseline artifacts and reports saved to Google Drive.")


recommendation_status | evaluated rows: 789,770
Accuracy: 0.8098 | Balanced accuracy: 0.6269
                 precision  recall  f1-score   support
recommended         0.9256  0.9032    0.9142  624568.0
not_recommended     0.4144  0.8877    0.5650   75961.0
no_idea             0.4555  0.0898    0.1500   89241.0
macro avg           0.5985  0.6269    0.5431  789770.0

Confusion matrix (rows: actual, columns: predicted)
                 recommended  not_recommended  no_idea
recommended           564091            52148     8329
not_recommended         7283            67428     1250
no_idea                38085            43143     8013

rating_band | evaluated rows: 925,196
Accuracy: 0.6330 | Balanced accuracy: 0.5691
           precision  recall  f1-score   support
low           0.4603  0.5138    0.4856  163553.0
medium        0.3450  0.4901    0.4049  159968.0
high          0.8212  0.7033    0.7577  601675.0
macro avg     0.5421  0.5691    0.5494  925196.0

Confusion matrix (rows: actu